In [ ]:
!pip install -q langdetect
!pip install -q spacymoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.3 MB/s eta 0:00:00


In [ ]:
# General imports
import numpy as np
import pandas as pd
from time import sleep
import os
from tqdm.auto import tqdm

# Translation imports
!pip install -q deep-translator
from deep_translator import GoogleTranslator
import nltk
nltk.download('punkt_tab')
from langdetect import detect

# Custom objects imports
from libraries.text_analyzer import TextAnalyzer

# Addresses
filtered_address = 'datasets/filtered'
unified_address = 'datasets/unified'
translated_address = 'datasets/translated'
joined_address = 'datasets/joined'
lastfilter_address = 'datasets/lastfilter'
datasets_address = 'datasets'

# Parameters
columns_initial = ['upvotes', 'date', 'user', 'content', 'subreddit']
columns_final = {0:'upvotes', 1:'date', 2:'user', 3:'content', 4:'subreddit', 5:'translated'}
keywords = ['job', 'work', 'profession', 'career']

# CSVs
programmers_csv = ['coding_comments', 'computerscience_comments', 'programming_comments']
designers_csv = ['Design_comments', 'GraphicDesigning_comments', 'graphic_design_comments']
all_csv = programmers_csv + designers_csv

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Description
Previously, to obtain the filtered datasets, we used those from the [Pushshift Dumps of Reddit from 2005 to 2024 from Watchful1](https://academictorrents.com/details/30dee5f0406da7a353aff6a8caa2d54fd01f2ca1), filtered through the [scripts created by Watchful1](https://github.com/Watchful1/PushshiftDumps/tree/master?tab=readme-ov-file) to translate from zst file format to csv file format.

Additionally, we also scraped data directly from reddit using the PRAW API, obtaining all possible posts and then all the comments, with the latter being filtered through the same keywords as the other datasets.

We decided to use only comments, since the number of posts was too little to be significant. Moreover, the content in posts was frequently only a single link, even though the reddit post also had additional text other than the media content from the link.

Specifically, the subreddits were:
- for programmers: r/coding, r/computerscience, r/programming
- for designers: r/Design, r/GraphicDesigning, r/graphic_design

To filter them, we used the keywords " AI ", "artificial intelligence", "chatgpt" and "chat gpt".

In the CSV files, we have that the columns are:

    upvotes,date,user,subreddit,content

where:
- upvotes are a number which defines how liked a comment is (it can go negative);
- date is when the comment was posted;
- user is who posted the comment;
- subreddit is the subreddit to which the comment was posted;
- content is the actual content of the comment.

## Quantities

	Programmers (torrented + scraped)
		coding            = 668 		+ 580
		computerscience   = 2'837 		+ 4'154
		programming       = 25'256 		+ 29'994
		------------------------------------------------------
		TOT               = 28'761		+ 34'728	= 63'489

	Designers (comments)
		Design            = 3'287		+ 6'266
		GraphicDesigning  =	323			+ 2'697
		graphic_design    = 18'814		+ 14'400
		------------------------------------------------------
		TOT               = 22'424		+ 23'363	= 45'787

    TOT_DATA = 109'276

## Adapt scraped comments to torrented datasets

Since some comments came from scraping directly from reddit and other comments came from torrenting, we need to make them uniform before applying any manipulation.

In [ ]:
def unify_scraped_torrented(df_scraped, df_torrented, columns, subreddit):
  columns_torrented = ['upvotes', 'date', 'user', 'content']
  columns_scraped = ['ups', 'created_datetime', 'author', 'comment_body']

  df_unified = pd.DataFrame(columns = columns)

  # remove useless columns from the torrented dataset
  for c in df_torrented.columns:
    if c not in columns_torrented:
      df_torrented.drop(c, axis = 1, inplace = True)

  # remove useless columns from the scraped dataset
  for c in df_scraped.columns:
    if c not in columns_scraped:
      df_scraped.drop(c, axis = 1, inplace = True)
    if c == 'created_datetime':
      df_scraped[c] = pd.to_datetime(df_scraped[c]).dt.date

  # combine the datasets into one
  for i in range(len(columns)-1):
    df_unified[columns[i]] = pd.concat([df_torrented[columns_torrented[i]], df_scraped[columns_scraped[i]]])

  df_unified['subreddit'] = subreddit

  print(f"{subreddit} -> {df_torrented.shape[0]} + {df_scraped.shape[0]} = {df_unified.shape[0]}")

  return df_unified

In [ ]:
for csv in all_csv:
  if not os.path.exists(unified_address):
    os.makedirs(unified_address)

  df_scraped = pd.read_csv(f"{filtered_address}/{csv}_scraped.csv")
  df_torrented = pd.read_csv(f"{filtered_address}/{csv}.csv")
  subreddit = (csv.split('_comments'))[0]
  df_unified = unify_scraped_torrented(df_scraped, df_torrented, columns_initial, subreddit)
  df_unified.to_csv(f"{unified_address}/{csv}.csv", index=False)

coding -> 668 + 580 = 1248
computerscience -> 2837 + 4154 = 6991
programming -> 25256 + 29994 = 55250
Design -> 3287 + 6266 = 9553
GraphicDesigning -> 323 + 2697 = 3020
graphic_design -> 18814 + 14400 = 33214


## Translation
Let's translate all comments from any language to english (if needed), by adding a new column 'translated', which contains the translated text.

In [ ]:
def translate_comments(comments, language, language_token):
    # initialize the Translator object
    gt = GoogleTranslator(source='auto', target=language)
    comments.reset_index(drop=True, inplace=True)

    # translate
    for i in tqdm(range(len(comments))):
        text = comments.loc[i,'content']
        try:
            # if the text is english, then it is not needed to translate
            if detect(text) == 'en':
                comments.loc[i,'translated'] = text
                continue
        except:
            pass

        comments.loc[i,'translated'] = None

        # to work around the limit of 5000 characters per request
        if len(text) > 5000:
            # tokenize the sentence (split it)
            token = nltk.sent_tokenize(text, language_token)
            for tt in token:
                # translate the tokens
                translated_text = gt.translate(tt)
                if comments.loc[i,'translated'] is None:
                    comments.loc[i,'translated'] = translated_text
                elif translated_text is not None:
                    comments.loc[i,'translated'] = comments.loc[i,'translated'] + translated_text

                # to work around the limit of 5 requests per second
                sleep(0.5)
        else:
            translated_text = gt.translate(text)
            comments.loc[i,'translated'] = translated_text

    comments.dropna(inplace=True)
    return comments

In [ ]:
# create the translated directory
if not os.path.exists(translated_address):
    os.makedirs(translated_address)

for csv in all_csv:
    df = pd.read_csv(f'{unified_address}/{csv}.csv')

    # translate all the comments in the CSV
    translated_df = translate_comments(df, 'en', 'english')
    translated_df.to_csv(f'{translated_address}/{csv}.csv', index=False)

  0%|          | 0/1248 [00:00<?, ?it/s]

  0%|          | 0/6991 [00:00<?, ?it/s]

  0%|          | 0/55250 [00:00<?, ?it/s]

  0%|          | 0/9553 [00:00<?, ?it/s]

  0%|          | 0/3020 [00:00<?, ?it/s]

  0%|          | 0/33214 [00:00<?, ?it/s]

## Join CSVs
At this point, in the CSV files the columns are:

    upvotes,date,user,content,subreddit,translated

Here, we join all files which contain subreddit comments related to a specific category, obtaining two files, one for programmers and one for designers. We also joined those two files to create a single file containing all of the comments.

In [ ]:
def join_csv(base_address, all_csv, columns):
    joined = None
    for csv in all_csv:
        df = pd.read_csv(f'{base_address}/{csv}.csv', header=None)
        # join CSVs
        if joined is None:
            joined = df
        else:
            joined = pd.concat([joined, df])

    joined.rename(columns=columns, inplace=True)
    joined.drop_duplicates()
    return joined

In [ ]:
programmers = join_csv(translated_address, programmers_csv, columns_final)
designers = join_csv(translated_address, designers_csv, columns_final)

if not os.path.exists(joined_address):
    os.makedirs(joined_address)

programmers.to_csv(f'{joined_address}/comments_programmers.csv', index=False)
designers.to_csv(f'{joined_address}/comments_designers.csv', index=False)

## Last Filtering
To filter for only the comments related to jobs, we use the keywords "job", "work", "profession" and "career". These will allow us to obtain the csv files which will actually be used for the analysis, which are:
- for programmers: programmers.csv
- for designers: designers.csv
- for all comments: all_comments.csv

In [ ]:
def filter_comments(comments, keywords):
    df = None
    # filter comments by keywords
    for k in keywords:
        f_df = comments.loc[comments['translated'].str.contains(k)]
        if df is None:
            df = f_df
        else:
            df = pd.concat([df, f_df])

    df = df.drop_duplicates()
    return df

In [ ]:
programmers = pd.read_csv(f'{joined_address}/comments_programmers.csv')
designers = pd.read_csv(f'{joined_address}/comments_designers.csv')

filtered_programmers = filter_comments(programmers, keywords)
filtered_designers = filter_comments(designers, keywords)
filtered_comments = pd.concat([filtered_programmers, filtered_designers])

if not os.path.exists(lastfilter_address):
    os.makedirs(lastfilter_address)

# save to CSV
filtered_programmers.to_csv(f'{lastfilter_address}/comments_programmers.csv', index=False)
filtered_designers.to_csv(f'{lastfilter_address}/comments_designers.csv', index=False)

print(f"Programmers comments: {filtered_programmers.shape[0]}")
print(f"Designers comments: {filtered_designers.shape[0]}")
print(f"Total comments: {filtered_comments.shape[0]}")

Programmers comments: 18320
Designers comments: 17856
Total comments: 36176


## Text Cleaning
To clean text, we created an object called TextAnalyzer.

In [ ]:
programmers = pd.read_csv(f'{lastfilter_address}/comments_programmers.csv')
designers = pd.read_csv(f'{lastfilter_address}/comments_designers.csv')

column_to_clean = "translated"
analyzer = TextAnalyzer()
analyzer.add_clean_text(programmers, column_to_clean)
analyzer.add_clean_text(designers, column_to_clean)
comments = pd.concat([programmers, designers])

# save to CSV
programmers.to_csv(f'{datasets_address}/programmers.csv', index=False)
designers.to_csv(f'{datasets_address}/designers.csv', index=False)

Processing texts: 100%|██████████| 17856/17856 [03:14<00:00, 91.98it/s] 
